In [11]:
##Carga datos y elimina duplicados
import pandas as pd

df = pd.read_csv("../data/coah.csv")
print("Registros originales:", len(df))

df = df.drop_duplicates(subset="texto").reset_index(drop=True)
print("Registros sin duplicados:", len(df)) 

Registros originales: 1816
Registros sin duplicados: 1792


In [12]:
##Division entrenamiento/prueba
from sklearn.model_selection import train_test_split

X = df["texto"]          # características: texto de la reseña
y = df["sentimiento"]    # etiqueta: negativo, neutro o positivo

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print(f"Entrenamiento: {len(X_train)} | Prueba (bajo llave): {len(X_test)}") 

Entrenamiento: 1433 | Prueba (bajo llave): 359


In [13]:
##Comprobacion // Las proporciones de clase se conservan
proporciones = pd.DataFrame({
    "completo": y.value_counts(normalize=True),
    "train": y_train.value_counts(normalize=True),
    "test": y_test.value_counts(normalize=True),
}).round(3)
print(proporciones)

# Ningún texto debe estar en ambos conjuntos
assert set(X_train).isdisjoint(set(X_test)), "Hay textos repetidos entre train y test"
print("\nOK: ningún texto aparece en entrenamiento y prueba a la vez.")

             completo  train   test
sentimiento                        
positivo        0.562  0.562  0.563
negativo        0.279  0.279  0.279
neutro          0.159  0.159  0.159

OK: ningún texto aparece en entrenamiento y prueba a la vez.


In [ ]:
##Vectorizacion TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Dimensiones del entrenamiento:", X_train_tfidf.shape)
print("Dimensiones de la prueba:", X_test_tfidf.shape)

Dimensiones del entrenamiento: (1433, 13786)
Dimensiones de la prueba: (359, 13786)


In [15]:
##Tratamiento del desbalance de clases
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, f1_score, recall_score

scoring = {
    "f1_macro": "f1_macro",
    "f1_neutro": make_scorer(f1_score, labels=["neutro"], average="macro"),
    "recall_neutro": make_scorer(recall_score, labels=["neutro"], average="macro"),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

filas = []
for nombre, pesos in [("Sin tratamiento", None), ("class_weight='balanced'", "balanced")]:
    pipe = make_pipeline(TfidfVectorizer(),
                         LogisticRegression(max_iter=1000, class_weight=pesos, random_state=42))
    r = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    filas.append({
        "Estrategia": nombre,
        "F1 macro": f"{r['test_f1_macro'].mean():.3f} ± {r['test_f1_macro'].std():.3f}",
        "F1 neutro": round(r["test_f1_neutro"].mean(), 3),
        "Recall neutro": round(r["test_recall_neutro"].mean(), 3),
    })

resultados = pd.DataFrame(filas)
resultados

,Estrategia,F1 macro,F1 neutro,Recall neutro
0,Sin tratamiento,0.590 ± 0.023,0.104,0.057
1,class_weight='balanced',0.714 ± 0.040,0.464,0.461
